In [1]:
-- 02_silver_clean - Cell 1: pre-flight check on bronze
SELECT 
    COUNT(*)                              AS bronze_rows,
    MIN(tpep_pickup_datetime)             AS earliest_pickup,
    MAX(tpep_pickup_datetime)             AS latest_pickup
FROM bronze_yellow_trips_raw;

StatementMeta(, 0aaeef9e-393e-4789-9447-598414f189ee, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 3 fields>

In [2]:
-- 02_silver_clean - Cell 2: build silver_yellow_trips_clean
CREATE OR REPLACE TABLE silver_yellow_trips_clean
USING DELTA
PARTITIONED BY (year, month)
AS
WITH base AS (
    -- Step 1: rename to snake_case + unit suffixes, apply quality filters
    SELECT
        VendorID                              AS vendor_id,
        tpep_pickup_datetime                  AS pickup_datetime,
        tpep_dropoff_datetime                 AS dropoff_datetime,
        YEAR(tpep_pickup_datetime)            AS year,
        MONTH(tpep_pickup_datetime)           AS month,
        passenger_count                       AS passenger_count,
        trip_distance                         AS trip_distance_mi,
        PULocationID                          AS pu_location_id,
        DOLocationID                          AS do_location_id,
        RatecodeID                            AS ratecode_id,
        payment_type                          AS payment_type_id,
        store_and_fwd_flag                    AS store_and_fwd_flag,
        fare_amount                           AS fare_amount,
        extra                                 AS extra_amount,
        mta_tax                               AS mta_tax,
        tip_amount                            AS tip_amount,
        tolls_amount                          AS tolls_amount,
        improvement_surcharge                 AS improvement_surcharge,
        total_amount                          AS total_amount,
        congestion_surcharge                  AS congestion_surcharge,
        airport_fee                           AS airport_fee,
        (UNIX_TIMESTAMP(tpep_dropoff_datetime) - 
         UNIX_TIMESTAMP(tpep_pickup_datetime)) / 60.0  AS duration_min
    FROM bronze_yellow_trips_raw
    WHERE tpep_pickup_datetime >= '2022-01-01'                    -- filtra vazamentos pré-2022
      AND tpep_pickup_datetime <  '2025-01-01'                    -- filtra vazamentos 2025/2026
      AND trip_distance > 0                                       -- distância tem que ser positiva
      AND fare_amount  >= 0                                       -- tarifa não pode ser negativa
      AND tpep_dropoff_datetime > tpep_pickup_datetime            -- dropoff > pickup
),
enriched AS (
    -- Step 2: derive operational fields
    SELECT
        base.*,
        CASE 
            WHEN duration_min > 0 THEN trip_distance_mi / (duration_min / 60.0)
            ELSE NULL
        END                                            AS avg_speed_mph,
        HOUR(pickup_datetime)                          AS pickup_hour,
        DAYOFWEEK(pickup_datetime)                     AS pickup_dow,
        DATE_FORMAT(pickup_datetime, 'EEEE')           AS pickup_day_name,
        CASE WHEN DAYOFWEEK(pickup_datetime) IN (1, 7) 
             THEN 1 ELSE 0 END                         AS is_weekend,
        CASE
            WHEN HOUR(pickup_datetime) BETWEEN 0  AND 5  THEN 'Late Night'
            WHEN HOUR(pickup_datetime) BETWEEN 6  AND 11 THEN 'Morning'
            WHEN HOUR(pickup_datetime) BETWEEN 12 AND 17 THEN 'Afternoon'
            ELSE 'Evening'
        END                                            AS day_part
    FROM base
)
-- Step 3: flag anomalies (don't discard — mark for analysis in DQ page)
SELECT
    *,
    CASE
        WHEN duration_min     > 360 THEN 1                -- > 6h trip = suspeito
        WHEN trip_distance_mi > 200 THEN 1                -- > 200mi = improvável dentro do estado
        WHEN avg_speed_mph    > 80  THEN 1                -- > 80mph = impossível em NYC urbana
        WHEN total_amount     > 1000 THEN 1               -- > $1000 = quase certo erro
        ELSE 0
    END AS is_anomalous
FROM enriched;

StatementMeta(, 0aaeef9e-393e-4789-9447-598414f189ee, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [3]:
-- 02_silver_clean - Cell 3: bronze vs silver volume + % discarded
SELECT
    (SELECT COUNT(*) FROM bronze_yellow_trips_raw)              AS bronze_rows,
    (SELECT COUNT(*) FROM silver_yellow_trips_clean)            AS silver_rows,
    (SELECT COUNT(*) FROM bronze_yellow_trips_raw) -
    (SELECT COUNT(*) FROM silver_yellow_trips_clean)            AS rows_discarded,
    ROUND(
        ((SELECT COUNT(*) FROM bronze_yellow_trips_raw) -
         (SELECT COUNT(*) FROM silver_yellow_trips_clean)) * 100.0 / 
        (SELECT COUNT(*) FROM bronze_yellow_trips_raw)
    , 4)                                                        AS pct_discarded;

StatementMeta(, 0aaeef9e-393e-4789-9447-598414f189ee, 4, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [4]:
-- 02_silver_clean - Cell 4: discard reason breakdown (sanity check)
SELECT
    SUM(CASE WHEN tpep_pickup_datetime < '2022-01-01' 
              OR tpep_pickup_datetime >= '2025-01-01'        THEN 1 ELSE 0 END) AS out_of_temporal_range,
    SUM(CASE WHEN trip_distance <= 0                         THEN 1 ELSE 0 END) AS zero_or_negative_distance,
    SUM(CASE WHEN fare_amount < 0                            THEN 1 ELSE 0 END) AS negative_fare,
    SUM(CASE WHEN tpep_dropoff_datetime <= tpep_pickup_datetime THEN 1 ELSE 0 END) AS invalid_duration
FROM bronze_yellow_trips_raw;

StatementMeta(, 0aaeef9e-393e-4789-9447-598414f189ee, 5, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [5]:
-- 02_silver_clean - Cell 5: anomaly breakdown
SELECT
    is_anomalous,
    COUNT(*)                                                          AS rows,
    ROUND(COUNT(*) * 100.0 / 
        (SELECT COUNT(*) FROM silver_yellow_trips_clean), 4)          AS pct
FROM silver_yellow_trips_clean
GROUP BY is_anomalous
ORDER BY is_anomalous;

StatementMeta(, 0aaeef9e-393e-4789-9447-598414f189ee, 6, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 3 fields>

In [6]:
-- 02_silver_clean - Cell 6: monthly volume after cleansing
SELECT
    year,
    month,
    COUNT(*)                          AS trips,
    ROUND(AVG(duration_min), 2)       AS avg_duration_min,
    ROUND(AVG(trip_distance_mi), 2)   AS avg_distance_mi,
    ROUND(AVG(total_amount), 2)       AS avg_total_usd
FROM silver_yellow_trips_clean
GROUP BY year, month
ORDER BY year, month;

StatementMeta(, 0aaeef9e-393e-4789-9447-598414f189ee, 7, Finished, Available, Finished, False)

<Spark SQL result set with 36 rows and 6 fields>

In [7]:
-- 02_silver_clean - Cell 7: sample anomalous trips for audit
SELECT
    pickup_datetime,
    dropoff_datetime,
    duration_min,
    trip_distance_mi,
    avg_speed_mph,
    fare_amount,
    total_amount,
    pu_location_id,
    do_location_id
FROM silver_yellow_trips_clean
WHERE is_anomalous = 1
ORDER BY total_amount DESC
LIMIT 20;

StatementMeta(, 0aaeef9e-393e-4789-9447-598414f189ee, 8, Finished, Available, Finished, False)

<Spark SQL result set with 20 rows and 9 fields>